In [0]:
%sql
SELECT * FROM olist.bronze.customer;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
cust_df=spark.table('olist.bronze.customer')
cust_df_cleaned=cust_df.dropDuplicates()\
                .dropna(subset=['customer_id'])\
                .withColumn('customer_city',initcap(trim(col('customer_city'))))\
                .withColumn('customer_zip_code_prefix',lpad(col('customer_zip_code_prefix'),5,0))\
                .withColumn('Is_Cust_Repeat',count('customer_unique_id').over(Window.partitionBy('customer_unique_id'))>1)
cust_df_cleaned.write.mode('overWrite').saveAsTable('olist.silver.customers')


In [0]:
orders_df=spark.table('olist.bronze.orders')
clean_orders_df=orders_df.dropDuplicates()\
                .dropna(subset=['order_id'])\
                .withColumn('order_purchase_timestamp',to_timestamp(col('order_purchase_timestamp')))\
                .withColumn('order_approved_at',to_timestamp(col('order_approved_at')))\
                .withColumn('order_delivered_carrier_date',to_timestamp(col('order_delivered_carrier_date')))\
                .withColumn('order_delivered_customer_date',to_timestamp(col('order_delivered_customer_date')))\
                .withColumn('order_estimated_delivery_date',to_timestamp(col('order_estimated_delivery_date')))\
                .withColumn('order_status',upper(col('order_status')))\
                .withColumn('late_delivery', when((col('order_status')=='DELIVERED'),(col('order_estimated_delivery_date')<col('order_delivered_customer_date'))).otherwise(None))\
                .withColumn('delivery_days_estimated',datediff(col('order_estimated_delivery_date'),col('order_purchase_timestamp')))\
                .withColumn('delivery_days_actual',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))\
                .withColumn('delay_in_delivery',datediff('order_delivered_customer_date','order_estimated_delivery_date'))\
                .withColumn('order_year',year(col('order_purchase_timestamp')))
clean_orders_df.write.mode('overWrite').saveAsTable('olist.silver.orders')

In [0]:
orderitems_df=spark.table('olist.bronze.orderitems')
clean_orderitems_df=orderitems_df.dropDuplicates()\
                .dropna(subset=['order_id','order_item_id'])\
                .withColumn('shipping_limit_date',to_timestamp(col('shipping_limit_date')))\
                .withColumn('price',col('price').cast('double'))\
                .withColumn('freight_value',col('freight_value').cast('double'))\
                .withColumn('total_price',round(col('price')+col('freight_value'),2))\
                .withColumn('freight_pct',round(col('freight_value')/col('price')*100,2))\
                .withColumn('is_high_freight',when(col('freight_pct')>50,'Y').otherwise('N'))
clean_orderitems_df.write.mode('overWrite').saveAsTable('olist.silver.orderitems')

In [0]:
products_df=spark.table('olist.bronze.products')
product_df_cleaned=products_df.dropDuplicates()\
                .dropna(subset=['product_id'])\
                .join(spark.table('olist.bronze.olist_ctg_name_trans'),on='product_category_name',how='left')\
                .withColumn('category',coalesce(col('product_category_name_english'),col('product_category_name')))\
                .drop('product_category_name','product_category_name_english','_rescued_data')\
                .withColumn('product_name_lenght',col('product_name_lenght').cast('integer'))\
                .withColumn('product_description_lenght',col('product_description_lenght').cast('integer'))\
                .withColumn('product_photos_qty',col('product_photos_qty').cast('integer'))\
                .withColumn('product_weight_g',col('product_weight_g').cast('integer'))\
                .withColumn('product_length_cm',col('product_length_cm').cast('integer'))\
                .withColumn('product_height_cm',col('product_height_cm').cast('integer'))\
                .withColumn('product_width_cm',col('product_width_cm').cast('integer'))\
                .withColumn('product_volume_cm3',round(col('product_length_cm')*col('product_height_cm')*col('product_width_cm'),2))
product_df_cleaned.write.mode('overWrite').saveAsTable('olist.silver.products_cleaned')

In [0]:
payments_df=spark.table('olist.bronze.payments')
clean_payments_df=payments_df.dropDuplicates(subset=['order_id','payment_sequential'])\
                .dropna(subset=['order_id'])\
                .withColumn('payment_sequential',col('payment_sequential').cast('integer'))\
                .withColumn('payment_installments',col('payment_installments').cast('integer'))\
                .withColumn('payment_value',col('payment_value').cast('decimal(10,2)'))\
                .withColumn('payment_type',upper(trim(col('payment_type'))))\
                .withColumn('total_payment',sum('payment_value').over(Window.partitionBy('order_id')))\
                .withColumn('via_installments',when(col('payment_installments')>1,True).otherwise(False))
clean_payments_df.write.mode('overWrite').saveAsTable('olist.silver.payments')



In [0]:
%sql
SELECT * FROM olist.bronze.olist_geolocation;

In [0]:
loc_df=spark.table('olist.bronze.olist_geolocation')
clean_loc_df=loc_df.dropDuplicates()\
                .dropna(subset=['geolocation_zip_code_prefix'])\
                .withColumn('geolocation_zip_code_prefix',lpad(col('geolocation_zip_code_prefix'),5,'0'))\
                .withColumn('geolocation_lat',col('geolocation_lat').cast('double'))\
                .withColumn('geolocation_lng',col('geolocation_lng').cast('double'))\
                .withColumn('geolocation_city',initcap(col('geolocation_city')))\
                .withColumn('geolocation_state',upper(col('geolocation_state')))\
                .drop("_rescued_data")
clean_loc_df.write.mode('overWrite').saveAsTable('olist.silver.geolocation')